## Planning:
- Load data
- Maximum Parameter with Optuna
- Blending with Nelder Mead
- Output

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import roc_auc_score

# import numpy as np
from scipy.optimize import minimize
# from sklearn.metrics import roc_auc_score

# import OS
import os

In [2]:
train = pd.read_csv("data/train.csv")
train_data = train[["id","is_fraud"]]

test = pd.read_csv("data/test.csv")
test_data = test[["id"]]

submit = pd.read_csv("data/sample_submission.csv")

In [3]:
folder_oof = "./oof"
folder_sub = "./sub"

In [4]:
content_oof = os.listdir(folder_oof)
content_sub = os.listdir(folder_sub)

In [5]:
for oof in content_oof:
    path_oof = folder_oof+"/"+oof
    oof_data = pd.read_csv(path_oof)
    train_data = pd.concat([train_data,oof_data.iloc[:,[1]]],axis=1)

In [6]:
for sub in content_sub:
    path_sub = folder_sub+"/"+sub
    sub_data = pd.read_csv(path_sub)
    test_data = pd.concat([test_data,sub_data.iloc[:,[1]]],axis=1)

## Data Process for Modeling

In [7]:
X = train_data.drop(columns=["id","is_fraud"])
y = train_data["is_fraud"]
X_test = test_data.drop(columns=["id"])
X_test.columns = X.columns

In [8]:
'''
X = X.drop(columns=["log_5cv"])
X_test = X_test.drop(columns=["log_5cv"])
'''

'\nX = X.drop(columns=["log_5cv"])\nX_test = X_test.drop(columns=["log_5cv"])\n'

In [9]:
num_models = X.shape[1]

In [10]:
num_models

33

In [11]:
# 2. Define the objective function with a penalty constraint
def objective_slsqp(weights):
    blend = X.values @ weights
    return 1.0 - roc_auc_score(y.values, blend)

In [12]:
# 3. Setup and Run the Optimizer
# Start with equal weights for all 3 models
init_weights = np.array([1/num_models]*num_models)

In [13]:
# Define bounds (0 to 1 for each weight) and constraint (sum to 1)
bounds = [(0.0, 1.0) for _ in range(num_models)]
cons = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}

In [14]:
res = minimize(
    objective_slsqp, 
    init_weights, 
    method='SLSQP', 
    bounds=bounds, 
    constraints=cons,
    options={'maxiter': 500}
)

In [15]:
# 4. Process and Normalize the Final Best Weights
best_weights = res.x

In [16]:
# 5. Print Results
print("Optimization Successful:", res.success)
print(f"Optimized Blending Weights: {np.round(best_weights, 4)}")

Optimization Successful: True
Optimized Blending Weights: [0.0337 0.0439 0.0019 0.016  0.0483 0.0337 0.0439 0.0425 0.0382 0.0137
 0.0793 0.0198 0.0124 0.0095 0.0063 0.0124 0.0399 0.0259 0.     0.0188
 0.0451 0.0153 0.0411 0.0506 0.0862 0.0411 0.0411 0.0506 0.0062 0.0221
 0.0025 0.0398 0.0185]


In [17]:
# Calculate the actual best ROC-AUC score using the optimized weights
X_blend = X.values @ best_weights
best_auc = roc_auc_score(y.values, X_blend)

print(f"Best ROC-AUC Score: {best_auc:.4f}")

Best ROC-AUC Score: 0.9464


In [18]:
# Compute the final blended predictions
final_blend = X_test.values @ best_weights

In [19]:
submit["is_fraud"] = final_blend

In [20]:
submit.to_csv("Blending SLSQP v2.csv",index=False)